# Intensity datasets – EDA, cleaning & dashboard planning
Files: `dailyIntensities`, `hourlyIntensities`, `minuteIntensitiesNarrow`, `minuteIntensitiesWide`

Notes:
- All use the same scale: 0 = sedentary, 1 = light, 2 = moderate and 3 = vigorous.
- Daily gives minutes in each band per day, plus distance covered at each intensity. 
- Hourly gives the intensity score (0–180) for each hour, so you can see when in the day people are active.
- Minute (narrow and wide) is the raw minute-by-minute stream. Narrow and wide are the same data, one long and one reshaped. 
- They show when people are active not just how much 

In [ ]:
# imports
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path

repo = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if p.name == "DATA2007-Fitbit-Dashboard-Group")
FOLDER = repo / "datafolder" / "FitBit Dataset-20261009"
DATETIME = "%m/%d/%Y %I:%M:%S %p"  # US format month first
DATE = "%m/%d/%Y"

In [ ]:
# csv
daily         = pd.read_csv(FOLDER / "dailyIntensities_merged.csv")
hourly        = pd.read_csv(FOLDER / "hourlyIntensities_merged.csv")
minute_narrow = pd.read_csv(FOLDER / "minuteIntensitiesNarrow_merged.csv")
minute_wide   = pd.read_csv(FOLDER / "minuteIntensitiesWide_merged.csv")

### 1. Format time columns types

In [ ]:
daily["ActivityDay"] = pd.to_datetime(daily["ActivityDay"], format=DATE)
hourly["ActivityHour"] = pd.to_datetime(hourly["ActivityHour"], format=DATETIME)
minute_narrow["ActivityMinute"] = pd.to_datetime(
    minute_narrow["ActivityMinute"], format=DATETIME
)
minute_wide["ActivityHour"] = pd.to_datetime(
    minute_wide["ActivityHour"], format=DATETIME
)

### 2. Overview of the data

In [ ]:
# combine datasets
datasets = {
    "daily": {"df": daily, "time_col": "ActivityDay"},
    "hourly": {"df": hourly, "time_col": "ActivityHour"},
    "minute narrow": {"df": minute_narrow, "time_col": "ActivityMinute"},
    "minute wide": {"df": minute_wide, "time_col": "ActivityHour"},
}

overview = pd.DataFrame(
    {
        "rows": {name: len(info["df"]) for name, info in datasets.items()},
        "columns": {name: info["df"].shape[1] for name, info in datasets.items()},
        "time column": {name: info["time_col"] for name, info in datasets.items()},
    }
)
print(overview)

### 3. Check for missing values

In [ ]:
for name, info in datasets.items():
    missing_per_column = info["df"].isna().sum()  
    total_missing = missing_per_column.sum()

    print(f"{name:14s} total missing: {total_missing}")
    if total_missing > 0:  # only list columns that have a problem
        print(missing_per_column[missing_per_column > 0])

### 4. Do files cover the same people

In [ ]:
# The set of user IDs in the daily file is our reference
daily_users = set(daily["Id"])

# Compare every file's users to the reference
for name, info in datasets.items():
    users = set(info["df"]["Id"])
    print(f"{name:14s} {len(users)} users, same as daily? {users == daily_users}")

### 5. Data exploration

Understanding the categories

In [ ]:
minute_cols = [
    "SedentaryMinutes",
    "LightlyActiveMinutes",
    "FairlyActiveMinutes",
    "VeryActiveMinutes",
]
distance_cols = [
    "SedentaryActiveDistance",
    "LightActiveDistance",
    "ModeratelyActiveDistance",
    "VeryActiveDistance",
]

# New columns, one value per row (per person per day)
daily["TotalMinutes"] = daily[minute_cols].sum(axis=1)
daily["ActiveMinutes"] = (
    daily["LightlyActiveMinutes"]
    + daily["FairlyActiveMinutes"]
    + daily["VeryActiveMinutes"]
)
daily["ModVigMinutes"] = daily["FairlyActiveMinutes"] + daily["VeryActiveMinutes"]
daily["TotalActiveDistance"] = daily[distance_cols].sum(axis=1)

# Summary table: one row per column
summary = daily[minute_cols + distance_cols].describe().T.round(2)
print(summary)

##### NEEED TO UNDESTAND THIS OUTPUT TABLE BETTER - WHY IS THE COUNT ALWAYS 940?

How much was the tracker worn?

In [ ]:
FULL_DAY = 1440  # minutes in a day

full_days = (daily["TotalMinutes"] == FULL_DAY).sum()
partial_days = (daily["TotalMinutes"] < FULL_DAY).sum()
over_days = (daily["TotalMinutes"] > FULL_DAY).sum()
print("Full days:", full_days, " Partial days:", partial_days)

not_worn = daily["SedentaryMinutes"] == FULL_DAY
print(
    "Not-worn days:",
    not_worn.sum(),
    "across",
    daily.loc[not_worn, "Id"].nunique(),
    "users",
)

Proportion of the day by intensity level

In [ ]:
levels = ["Sedentary", "Light", "Moderate", "Vigorous"]
intensity_names = dict(enumerate(levels))  # 0 -> Sedentary, 1 -> Light, ...
daily_cols = {
    "SedentaryMinutes": "Sedentary",
    "LightlyActiveMinutes": "Light",
    "FairlyActiveMinutes": "Moderate",
    "VeryActiveMinutes": "Vigorous",
}

# Daily minutes with friendly column names, for all days and for worn days only
daily_minutes = daily.rename(columns=daily_cols)[levels]
worn_minutes = daily_minutes[daily["SedentaryMinutes"] != 1440]


def percent_share(minutes_table):
    """Each intensity level as a % of all recorded minutes."""
    totals = minutes_table.sum()
    return totals / totals.sum() * 100


summary = (
    pd.DataFrame(
        {
            "% of minutes": minute_narrow["Intensity"]
            .map(intensity_names)
            .value_counts(normalize=True)
            .mul(100),
            "% of time (daily, all days)": percent_share(daily_minutes),
            "% of time (daily, worn days)": percent_share(worn_minutes),
            "avg min/day (all days)": daily_minutes.mean(),
            "avg min/day (worn days)": worn_minutes.mean(),
            "median min/day (worn days)": worn_minutes.median(),
            "% worn days with 0 min": (worn_minutes == 0).mean() * 100,
        }
    )
    .loc[levels]
    .round(1)
)

summary.T  

# For the dashboard:

In [ ]:
daily["ModVigMinutes"] = daily["FairlyActiveMinutes"] + daily["VeryActiveMinutes"]
worn = daily[daily["SedentaryMinutes"] != 1440].copy()  # drop days the tracker wasn't worn

my_id = 1624580081     

## 1. “How did I do today and this week?” 
Show: moderate plus vigorous minutes today, this week, and against the 150-minute weekly target, as a progress bar.
Data: daily file.
Explore: how many users hit 150 a week, and how much that changes if you count vigorous minutes double (the WHO convention).

In [ ]:
worn["Week"] = worn["ActivityDay"].dt.to_period("W")

weekly = worn.groupby(["Id", "Week"]).agg(
    days      = ("ActivityDay", "count"),
    moderate  = ("FairlyActiveMinutes", "sum"),
    vigorous  = ("VeryActiveMinutes", "sum"),
).reset_index()

weekly["who_minutes"] = weekly["moderate"] + 2 * weekly["vigorous"]   # vigorous counts double
weekly["hit_target"]  = weekly["who_minutes"] >= 150

print(weekly[weekly["Id"] == my_id])

full_weeks = weekly[weekly["days"] >= 6]                              # ignore part-weeks
print("Full weeks hitting 150:", round(full_weeks["hit_target"].mean() * 100), "%")

## 2. “How do I compare to other users?”
Show: a leaderboard on a fair measure, and my position in the group distribution.
Data: daily file, with the non-wear and minimum-days filters.
Explore: which ranking measure is fair. Total active minutes rewards long, easy walks, while vigorous minutes rewards hard sessions. Letting the user toggle between them is a good dashboard feature.
Watch out: the data is skewed, so show medians. Users also have very different amounts of data (4 to 31 days), so show how many days each ranking is based on.

In [ ]:
MIN_DAYS = 7     # hide users with too little data

board = worn.groupby("Id").agg(
    days      = ("ActivityDay", "count"),
    mod_vig   = ("ModVigMinutes", "median"),
    vigorous  = ("VeryActiveMinutes", "median"),
)
board = board[board["days"] >= MIN_DAYS]

board["rank_mod_vig"]  = board["mod_vig"].rank(ascending=False, method="min").astype(int)
board["rank_vigorous"] = board["vigorous"].rank(ascending=False, method="min").astype(int)

print(board.sort_values("rank_mod_vig").head(5))
print("Rank agreement:", round(board["mod_vig"].corr(board["vigorous"], method="spearman"), 2))

## 3. “Am I improving?” 
Show: a line of daily active minutes with a 7-day average, with the group average for context.
Data: daily file.
Explore: whether a trend actually exists over a month, or whether it’s just noise. With 31 days, be careful about claiming one.

In [ ]:
my_days     = worn[worn["Id"] == my_id].set_index("ActivityDay")["ModVigMinutes"]
my_trend    = my_days.rolling(7, min_periods=4).mean()
group_trend = worn.groupby("ActivityDay")["ModVigMinutes"].mean().rolling(7, min_periods=4).mean()

ax = my_days.plot(alpha=0.3, label="me (daily)", figsize=(10, 3.5))
my_trend.plot(ax=ax, label="me (7-day avg)")
ax.legend(); plt.show()

## 4. “When do I train?”
Show: a heatmap of hour of day against weekday, for me and for the group.
Data: hourly file.
Explore: each person’s typical rhythm, such as morning versus evening exercisers. In the group, the evening peak (about 5–7pm) is the main feature. A question for friends: “who trains at the same time as me?”

In [ ]:
my_hours = hourly[hourly["Id"] == my_id].copy()
my_hours["Hour"]    = my_hours["ActivityHour"].dt.hour
my_hours["Date"]    = my_hours["ActivityHour"].dt.normalize()
my_hours["Weekday"] = my_hours["ActivityHour"].dt.day_name()

# ---------- Plot 1: daily habits (average hour-by-hour profile) ----------
by_hour = my_hours.groupby("Hour")["TotalIntensity"].mean().reindex(range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.fill_between(by_hour.index, by_hour.values, color="C1", alpha=0.35)
ax.plot(by_hour.index, by_hour.values, color="C1", lw=2)

busiest = by_hour.idxmax()
ax.annotate(f"Busiest hour: {busiest}:00", xy=(busiest, by_hour.max()),
            xytext=(busiest, by_hour.max() * 1.12), ha="center", arrowprops=dict(arrowstyle="-"))
ax.set_ylim(0, by_hour.max() * 1.3)
ax.set_xticks(range(0, 24, 3)); ax.set_xticklabels([f"{h}:00" for h in range(0, 24, 3)])
ax.set_xlabel("Hour of day"); ax.set_ylabel("Average intensity per hour")
ax.set_title("My daily habits")
plt.tight_layout(); plt.show()

# ---------- Plot 2: weekly habits (total activity for each day of the week) ----------
day_totals = my_hours.groupby("Date").agg(
    total=("TotalIntensity", "sum"),
    hours=("TotalIntensity", "size"),
    Weekday=("Weekday", "first"),
)
day_totals = day_totals[day_totals["hours"] == 24]            # only fully recorded days

order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday = day_totals.groupby("Weekday")["total"].mean().reindex(order)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(order, by_weekday.values, color="C1", alpha=0.6)
ax.set_ylabel("Average total intensity for the day")
ax.set_title("My weekly habits")
plt.tight_layout(); plt.show()

## 5. “Am I consistent?”
Show: how many days a week I was active, and the gap between my best and worst days.
Data: daily file.
Explore: the spread of daily activity per person. Two people with the same average can be very different: one steady, one weekend-only.

In [ ]:
ACTIVE_DAY = 30      # a day counts as "active" with 30+ hard minutes

consistency = worn.groupby("Id").agg(
    days        = ("ActivityDay", "count"),
    active_days = ("ModVigMinutes", lambda s: (s >= ACTIVE_DAY).sum()),
)
consistency = consistency[consistency["days"] >= 14]
consistency["pct_active_days"] = consistency["active_days"] / consistency["days"] * 100

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

mine         = daily[daily["Id"] == my_id].set_index("ActivityDay")
days         = pd.date_range(daily["ActivityDay"].min(), daily["ActivityDay"].max())
first_monday = days[0] - pd.Timedelta(days=days[0].dayofweek)
n_weeks      = (days[-1] - first_monday).days // 7 + 1

# One square per day: 0 = no data / not worn, 1 = worn but not active, 2 = active
grid = np.full((n_weeks, 7), np.nan)
fig, ax = plt.subplots(figsize=(8, 4))
for day in days:
    row, col = (day - first_monday).days // 7, day.dayofweek
    if day not in mine.index or mine.loc[day, "SedentaryMinutes"] == 1440:
        grid[row, col] = 0
    elif mine.loc[day, "ModVigMinutes"] >= ACTIVE_DAY:
        grid[row, col] = 2
    else:
        grid[row, col] = 1
    ax.text(col + 0.5, row + 0.5, day.day, ha="center", va="center", fontsize=9,
            color="white" if grid[row, col] == 2 else "#444")

colours = ListedColormap(["#e5e5e5", "#fde3c8", "#e8590c"])
ax.pcolormesh(np.ma.masked_invalid(grid), cmap=colours, vmin=0, vmax=2, edgecolors="white", linewidth=3)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.xaxis.tick_top(); ax.tick_params(length=0)
ax.set_xticks(np.arange(7) + 0.5); ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.set_yticks(np.arange(n_weeks) + 0.5)
ax.set_yticklabels([(first_monday + pd.Timedelta(weeks=w)).strftime("w/c %d %b") for w in range(n_weeks)])
for side in ax.spines.values(): side.set_visible(False)

ax.legend(handles=[Patch(color="#e8590c", label=f"Active ({ACTIVE_DAY}+ hard min)"),
                   Patch(color="#fde3c8", label="Worn, not active"),
                   Patch(color="#e5e5e5", label="No data / not worn")],
          loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False, fontsize=8)
ax.set_title(f"Active on {(grid == 2).sum()} of {(grid > 0).sum()} days worn", pad=28)
plt.tight_layout(); plt.show()

## Limitations

- Data quality: non-wear days and partial days should be filtered or flagged, so a day where the tracker was off doesn’t show as a lazy day.
- Small sample: 33 people for one month 